# 01 - Exploratory data analysis

Data: UCI *Default of Credit Card Clients* (Taiwan, 2005), loaded by the pipeline into PostgreSQL.
30,000 cardholders with a six-month history (Apr-Sep 2005) of statement balances, payments and repayment
status, plus whether they defaulted on the October payment.

This notebook looks at the data as it sits in the `core` schema and checks the assumptions the rest of the
project depends on.

In [1]:
import sys, json, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
from src.db import read_sql
from src.evaluation import plots  # applies the shared chart style
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
FIG = Path("../reports/figures")

## 1. Size and grain
`core.fact_account_month` should have exactly one row per customer per month.

In [2]:
read_sql('''SELECT (SELECT count(*) FROM core.dim_customer) AS customers,
                  (SELECT count(*) FROM core.fact_account_month) AS customer_months,
                  (SELECT avg(defaulted_next_month::int) FROM core.fact_default_outcome) AS oct_default_rate''')

,customers,customer_months,oct_default_rate
0,30000,180000,0.2212


## 2. Data-quality findings
Results of the validation step (full report in `docs/data_quality_report.md`). Nothing failed an ERROR check,
so no rows were quarantined. The warnings are mostly real customer behaviour (overpayment, over-limit) plus
two coding issues.

In [3]:
read_sql('''SELECT check_name, severity, rows_failed, pct_failed FROM audit.dq_check_results
            WHERE run_id = (SELECT max(run_id) FROM audit.pipeline_runs WHERE status = 'loaded')
              AND (rows_failed > 0) ORDER BY severity, rows_failed DESC''')

,check_name,severity,rows_failed,pct_failed
0,never_used_account,INFO,795,2.6500
1,over_limit_balance,WARN,3931,13.1030
2,delinquent_with_no_balance,WARN,2037,6.7900
3,credit_balance,WARN,1930,6.4330
4,balance_drop_exceeds_payment,WARN,612,2.0400
5,undocumented_education_code,WARN,345,1.1500
6,extreme_over_limit,WARN,343,1.1430
7,duplicate_profile,WARN,70,0.2330
8,undocumented_marriage_code,WARN,54,0.1800
9,payment_far_above_limit,WARN,4,0.0130


## 3. Which bill does a payment pay?
The UCI documentation lists bills and payments by month but doesn't say how they line up. If the payment in
month *t* pays the bill from month *t-1*, it should match that bill exactly more often (people paying in full).

In [4]:
read_sql('''SELECT avg((payment_amount = prev_bill_amount)::int) AS pays_previous_bill,
                  avg((payment_amount = bill_amount)::int)      AS pays_same_month_bill
           FROM core.fact_account_month WHERE month_index > 1 AND payment_amount > 0''')

,pays_previous_bill,pays_same_month_bill
0,0.1764,0.0264


Payments match the previous month's bill far more often, so I use
`new charges_t = bill_t - bill_(t-1) + payment_t`. This identity is what makes activity, purchases and
interest estimable at all.

## 4. Repayment status coding by month

In [5]:
s = read_sql('''SELECT month_index, status_code, count(*) n FROM core.fact_account_month
                  WHERE status_code BETWEEN -2 AND 3 GROUP BY 1, 2''')
s.pivot(index="status_code", columns="month_index", values="n").fillna(0).astype(int).rename(
    columns=dict(zip(range(1, 7), ["Apr", "May", "Jun", "Jul", "Aug", "Sep"])))

month_index,Apr,May,Jun,Jul,Aug,Sep
status_code,,,,,,
-2,4895,4546,4348,4085,3782,2759
-1,5740,5539,5687,5938,6050,5686
0,16286,16947,16455,15764,15730,14737
1,0,0,2,4,28,3688
2,2766,2626,3159,3819,3927,2667
3,184,178,180,240,326,322


Status 1 ("one month late") barely exists before September; in Apr-Aug accounts jump from 0 to 2.
In September it suddenly covers ~3,700 accounts. I treat September's codes as not comparable with the earlier
months: trend comparisons use 60+ DPD (status >= 2), and the dormancy model avoids raw status codes.

## 5. Portfolio trends

In [6]:
port = read_sql("SELECT * FROM mart.portfolio_monthly ORDER BY month_index")
port[["month_label", "active_rate", "total_balance", "portfolio_utilization", "paid_in_full_rate",
      "delinquency_rate_60plus", "new_60plus_accounts", "cured_from_60plus"]]

,month_label,active_rate,total_balance,portfolio_utilization,paid_in_full_rate,delinquency_rate_60plus,new_60plus_accounts,cured_from_60plus
0,Apr 2005,NaN,"1,168,268,063.0000",0.2325,NaN,0.1026,0,0
1,May 2005,0.8686,"1,210,412,763.0000",0.2409,0.2549,0.0989,862,973
2,Jun 2005,0.8783,"1,298,989,558.0000",0.2585,0.2420,0.1169,1254,713
3,Jul 2005,0.8894,"1,411,355,065.0000",0.2809,0.2466,0.1403,1623,921
4,Aug 2005,0.9014,"1,476,195,541.0000",0.2938,0.2581,0.1470,1479,1276
5,Sep 2005,0.9202,"1,537,381,257.0000",0.3060,0.2544,0.1043,991,447


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
m = port.month_label.str[:3]
axes[0].plot(m, port.total_balance / 1e9, marker="o"); axes[0].set_title("Balance outstanding (NT$ bn)")
axes[1].plot(m, port.portfolio_utilization * 100, marker="o"); axes[1].set_title("Portfolio utilization (%)")
axes[2].plot(m, port.delinquency_rate_60plus * 100, marker="o"); axes[2].set_title("60+ DPD rate (%), Sep not comparable")
plt.tight_layout(); plt.show()

Balances grew about 32% between April and September while 60+ DPD delinquency rose from ~10% to ~15%
(Aug). The data comes from the start of Taiwan's 2005-06 card-debt crisis, which fits this picture.

## 6. Default rate by observable characteristics

In [8]:
read_sql('''SELECT f.status_latest AS sep_status, count(*) customers, avg(o.defaulted_next_month::int) default_rate
           FROM features.customer_risk_features f JOIN core.fact_default_outcome o USING (customer_id)
           WHERE f.status_latest <= 4 GROUP BY 1 ORDER BY 1''')

,sep_status,customers,default_rate
0,-2,2759,0.1323
1,-1,5686,0.1678
2,0,14737,0.1281
3,1,3688,0.3395
4,2,2667,0.6914
5,3,322,0.7578
6,4,76,0.6842


In [9]:
from scipy.stats import chi2_contingency
d = read_sql('''SELECT c.limit_tier, c.age_band, c.education, o.defaulted_next_month::int y
               FROM core.dim_customer c JOIN core.fact_default_outcome o USING (customer_id)''')
for col in ["limit_tier", "age_band", "education"]:
    t = d.groupby(col).y.agg(customers="size", default_rate="mean")
    chi2, p, dof, _ = chi2_contingency(pd.crosstab(d[col], d.y))
    display(t); print(f"chi-square p-value for {col}: {p:.2e}")

,customers,default_rate
limit_tier,,
1. <50k,4311,0.3607
2. 50k-150k,10979,0.2520
3. 150k-300k,9593,0.1682
4. 300k+,5117,0.1368


chi-square p-value for limit_tier: 4.22e-198


,customers,default_rate
age_band,,
21-24,2685,0.2719
25-34,13011,0.2030
35-44,9018,0.2186
45-54,4233,0.2393
55+,1053,0.2669


chi-square p-value for age_band: 8.18e-18


,customers,default_rate
education,,
Graduate school,10585,0.1923
High school,4917,0.2516
Other,123,0.0569
University,14030,0.2373
Unknown,345,0.0754


chi-square p-value for education: 1.09e-33


Current repayment status is by far the strongest single signal. Default rate falls steadily with credit
limit (the bank gave larger limits to customers it judged safer). The differences across limit tiers and age
bands are very unlikely to be chance (tiny chi-square p-values), but they are descriptive, not causal.
Note that even customers who "paid duly" in September have a ~17% recorded default rate, so the label is broad:
it is a missed payment, not a write-off.

## 7. Roll rates and delinquency cohorts

In [10]:
r = read_sql('''SELECT from_bucket, to_bucket, sum(accounts) n FROM mart.roll_rates
               WHERE from_month_index <= 4 GROUP BY 1, 2''')
rr = r.pivot(index="from_bucket", columns="to_bucket", values="n").fillna(0)
(rr.div(rr.sum(axis=1), axis=0) * 100).round(1)

to_bucket,0. Current,1. 30 DPD,2. 60 DPD,3. 90+ DPD
from_bucket,,,,
0. Current,95.1000,0.0000,4.9000,0.0000
1. 30 DPD,0.0000,100.0000,0.0000,0.0000
2. 60 DPD,30.2000,0.0000,63.6000,6.1000
3. 90+ DPD,10.4000,0.0000,31.9000,57.7000


In [11]:
read_sql('''SELECT cohort_month, months_since_entry, accounts, share_still_60plus, share_cured, oct_default_rate
           FROM mart.delinquency_cohorts ORDER BY cohort_month_index, months_since_entry''')

,cohort_month,months_since_entry,accounts,share_still_60plus,share_cured,oct_default_rate
0,May 2005,0,862,1.0000,0.0000,0.4200
1,May 2005,1,862,0.5371,0.4629,0.4200
2,May 2005,2,862,0.4107,0.5893,0.4200
3,May 2005,3,862,0.4304,0.5684,0.4200
4,May 2005,4,862,0.2749,0.5070,0.4200
5,Jun 2005,0,1233,1.0000,0.0000,0.3982
6,Jun 2005,1,1233,0.5515,0.4477,0.3982
7,Jun 2005,2,1233,0.4112,0.5880,0.3982
8,Jun 2005,3,1233,0.2547,0.5255,0.3982
9,Jul 2005,0,1370,1.0000,0.0000,0.3810


Delinquency is sticky but not permanent: about 30% of 60 DPD accounts are current again the next month,
and more than half (56-59%) of each new 60+ DPD cohort has cured two months later. (The "30 DPD" row is tiny because of
the coding issue above.)

**Next:** feature engineering (`02_feature_engineering.ipynb`).